CANDITDATE ELIMINATION ALGORITHM

In [ ]:
values = {
    'Sky': ['Sunny', 'Cloudy', 'Rainy'],
    'AirTemp': ['Warm', 'Cold'],
    'Humidity': ['Normal', 'High'],
    'Wind': ['Strong', 'Weak'],
    'Water': ['Warm', 'Cool'],
    'Forecast': ['Same', 'Change']
}

map = {i: key for i, key in enumerate(values.keys())}

In [ ]:
#Check for Consistency
def consistent(hyp, inst, lbl):
  # Check if hyp covers inst
  hyp_covers_inst = True
  for i in range(len(hyp)):
    if(hyp[i] != "?" and hyp[i] != inst[i]):
      hyp_covers_inst = False
      break

  if lbl == "Yes":
    return hyp_covers_inst
  else:
    return not hyp_covers_inst

In [ ]:
# Specialization of G
def specializeG(hyp, inst, map, values):
  specs = []
  for i in range(len(hyp)):
    if hyp[i]=="?":
      for value in values[map[i]]:
        if value != inst[i]:
          new_hyp = hyp[:]
          new_hyp[i] = value
          specs.append(new_hyp)
  return specs

In [ ]:

def CEA(ex):

  num_attributes = len(ex[0][0])
  G = [["?" for _ in range(num_attributes)]]
  S = []

  # Find the first positive example to initialize S
  found_positive = False
  for inst_features, inst_lbl in ex:
      if inst_lbl == "Yes":
          S = [inst_features[:]] # Deep copy the list
          found_positive = True
          break
  if not found_positive:
      S = [['0'] * num_attributes] # Placeholder if no positive example is found initially

  exn = 0
  print("Initialization")
  print("S =", S)
  print("G =", G)
  print()

  for example_pair in ex:
    inst, lbl = example_pair
    exn+=1

    if lbl == "Yes":
      G = [h for h in G if consistent(h,inst,lbl)]

      # Original block for generalizing S if S[0] is not consistent with positive example
      if S and not consistent(S[0],inst,lbl):
        temp_s = S[0][:]
        for i in range(len(temp_s)):
          if temp_s[i] == "0":
            temp_s[i] = inst[i]
          elif temp_s[i] != inst[i]:
            temp_s[i] = "?"
        S[0] = temp_s

    else:

      S = [h for h in S if consistent(h,inst,lbl)]

      if G:
        if not consistent(G[0],inst,lbl):
          newG = []
          for g in G:
            if not consistent(g, inst, lbl):
                newG.extend(specializeG(g,inst,map,values))
            else:
                newG.append(g)
          G = newG


    for prev_example_pair in ex[:exn]:
      prevInst, prevLbl = prev_example_pair
      if prevLbl=="Yes":
        G = [h for h in G if consistent(h,prevInst,prevLbl)]
        S = [h for h in S if consistent(h,prevInst,prevLbl)]
      else:
        G = [h for h in G if consistent(h,prevInst,prevLbl)]
        S = [h for h in S if consistent(h,prevInst,prevLbl)]

    print("S =", S)
    print("G =", G)
    print()

  return S, G

In [ ]:
training_examples = [
        (['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'], 'Yes'),
        (['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'], 'Yes'),
        (['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'], 'No'),
        (['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change'], 'Yes')]

GB,SB = CEA(training_examples)
print("Final Version Space:\n")
print("General Hypothesis:", GB)
print("Specific Hypothesis:", SB)

Initialization
S = [['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']]
G = [['?', '?', '?', '?', '?', '?']]

S = [['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']]
G = [['?', '?', '?', '?', '?', '?']]

S = [['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']]
G = [['?', '?', '?', '?', '?', '?']]

S = [['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']]
G = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']]

S = [['Sunny', 'Warm', '?', 'Strong', '?', '?']]
G = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?']]

Final Version Space:

General Hypothesis: [['Sunny', 'Warm', '?', 'Strong', '?', '?']]
Specific Hypothesis: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?']]
